# 🏛️ Lesson 1: Binary Search and Complexity Analysis

## Overview
* **Focus**: Learning how to systematically dismantle a problem using the 6-Step Method.
* **Core Algorithms**: Linear Search and Binary Search.
* **Primary Metric**: Time and Space Complexity ($O(N)$ vs. $O(\log N)$).

In [1]:
# Import testing tools and mathematical libraries
import math
from timeit import default_timer as timer

# Quick environment sanity check
math.sqrt(49)

7.0

## 1. The Core Problem
> **QUESTION 1**: Alice has some cards with numbers written on them. She arranges the cards in decreasing order and lays them out face down in a sequence on a table. She challenges Bob to pick out the card containing a given number by turning over as few cards as possible. Write a function to help Bob locate the card.

### Abstracting the Problem
* **Input**:
  * `cards`: A list of numbers sorted in decreasing order (e.g., `[13, 11, 10, 7, 4, 3, 1, 0]`)
  * `query`: The target number whose position needs to be found (e.g., `7`)
* **Output**:
  * `position`: The 0-indexed index of `query` inside `cards` (e.g., `3`). Return `-1` if the number is not found.

In [2]:
def locate_card(cards, query):
    pass

## 2. Test Cases and Edge Cases
To ensure correctness, we must evaluate our upcoming implementations against diverse scenarios, specifically tracking boundary constraints.

In [3]:
tests = []

# 1. Query occurs in the middle
tests.append({
    'input': {'cards': [13, 11, 10, 7, 4, 3, 1, 0], 'query': 7},
    'output': 3
})

# 2. Query is near the end
tests.append({
    'input': {'cards': [13, 11, 10, 7, 4, 3, 1, 0], 'query': 1},
    'output': 6
})

# 3. Query is the first element
tests.append({
    'input': {'cards': [4, 2, 1, -1], 'query': 4},
    'output': 0
})

# 4. Query is the last element
tests.append({
    'input': {'cards': [3, -1, -9, -127], 'query': -127},
    'output': 3
})

# 5. Cards contains exactly one element (the query)
tests.append({
    'input': {'cards': [6], 'query': 6},
    'output': 0
})

# 6. Edge Case: Query does not exist in the list
tests.append({
    'input': {'cards': [9, 7, 5, 2, -9], 'query': 4},
    'output': -1
})

# 7. Edge Case: Cards list is completely empty
tests.append({
    'input': {'cards': [], 'query': 7},
    'output': -1
})

# 8. Cards list contains repeating numbers, query is unique
tests.append({
    'input': {'cards': [8, 8, 6, 6, 6, 6, 6, 3, 2, 2, 2, 0, 0, 0], 'query': 3},
    'output': 7
})

# 9. Edge Case: Query occurs multiple times (should return the FIRST occurrence)
tests.append({
    'input': {'cards': [8, 8, 6, 6, 6, 6, 6, 6, 3, 2, 2, 2, 0, 0, 0], 'query': 6},
    'output': 2
})

## 3. Automation Testing Framework
Below are the local helper functions used to measure execution runtime and evaluate correctness metrics across our test suite.

In [4]:
def evaluate_test_cases(function, test_cases):
    total = len(test_cases)
    passed = 0
    
    for i, test_case in enumerate(test_cases):
        inputs = test_case['input']
        expected = test_case['output']
        
        start = timer()
        try:
            actual = function(**inputs)
            error = None
        except Exception as e:
            actual = None
            error = type(e).__name__
        end = timer()
        
        runtime = math.ceil((end - start) * 1e6) / 1000 # convert to ms
        
        if error:
            print(f"TEST CASE #{i}: FAILED with Exception [{error}]")
        elif actual == expected:
            print(f"TEST CASE #{i}: PASSED | Runtime: {runtime} ms")
            passed += 1
        else:
            print(f"TEST CASE #{i}: FAILED | Expected {expected}, got {actual}")
            
    print(f"\nSUMMARY: TOTAL: {total} | PASSED: {passed} | FAILED: {total - passed}")

## 4. Approach 1: Linear Search (Brute Force)
* **Strategy**: Iterate over the list sequentially from index 0 to the end, evaluating each element one-by-one.
* **Complexity**:
  * **Time Complexity**: $O(N)$ (Worst case requires examining every single element).
  * **Space Complexity**: $O(1)$ (Operates entirely in-place with a single integer pointer).

In [5]:
def locate_card_linear(cards, query):
    position = 0
    # Guard loop boundary via array size check to gracefully handle empty inputs
    while position < len(cards):
        if cards[position] == query:
            return position
        position += 1
    return -1

# Verify correctness across the entire test suite
evaluate_test_cases(locate_card_linear, tests)

TEST CASE #0: PASSED | Runtime: 0.004 ms
TEST CASE #1: PASSED | Runtime: 0.003 ms
TEST CASE #2: PASSED | Runtime: 0.001 ms
TEST CASE #3: PASSED | Runtime: 0.002 ms
TEST CASE #4: PASSED | Runtime: 0.002 ms
TEST CASE #5: PASSED | Runtime: 0.002 ms
TEST CASE #6: PASSED | Runtime: 0.001 ms
TEST CASE #7: PASSED | Runtime: 0.002 ms
TEST CASE #8: PASSED | Runtime: 0.002 ms

SUMMARY: TOTAL: 9 | PASSED: 9 | FAILED: 0


## 5. Approach 2: Binary Search (Optimized)
* **Strategy**: Leverage the sorted arrangement. Constantly check the midpoint of the bounds, eliminating half of the search space on each comparison.
* **Handling Duplicates**: When `cards[mid] == query`, peek to the left (`mid - 1`). If the previous element is also equal to the query, continue scanning leftward to locate the true first occurrence.
* **Complexity**:
  * **Time Complexity**: $O(\log N)$ (The input bounds shrink exponentially).
  * **Space Complexity**: $O(1)$ (Calculations are confined to primitive index tracking variables).

In [6]:
def test_location(cards, query, mid):
    if cards[mid] == query:
        # If true first occurrence lies further to the left, target the left boundary bounds
        if mid - 1 >= 0 and cards[mid - 1] == query:
            return 'left'
        return 'found'
    elif cards[mid] < query:
        return 'left' # Search left half due to descending data sorting order
    else:
        return 'right' # Search right half

def locate_card_binary(cards, query):
    lo, hi = 0, len(cards) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        result = test_location(cards, query, mid)
        
        if result == 'found':
            return mid
        elif result == 'left':
            hi = mid - 1
        elif result == 'right':
            lo = mid + 1
    return -1

# Verify correctness across the entire test suite
evaluate_test_cases(locate_card_binary, tests)

TEST CASE #0: PASSED | Runtime: 0.003 ms
TEST CASE #1: PASSED | Runtime: 0.005 ms
TEST CASE #2: PASSED | Runtime: 0.003 ms
TEST CASE #3: PASSED | Runtime: 0.003 ms
TEST CASE #4: PASSED | Runtime: 0.003 ms
TEST CASE #5: PASSED | Runtime: 0.003 ms
TEST CASE #6: PASSED | Runtime: 0.001 ms
TEST CASE #7: PASSED | Runtime: 0.003 ms
TEST CASE #8: PASSED | Runtime: 0.003 ms

SUMMARY: TOTAL: 9 | PASSED: 9 | FAILED: 0


## 6. Empirical Benchmarking: $O(N)$ vs. $O(\log N)$
We will evaluate both search architectures against a structural performance check featuring an inverse array containing **10 million items**.

In [7]:
# Generate large scale list descending from 10,000,000 down to 1
large_test_cards = list(range(10000000, 0, -1))
target_query = 2

# 1. Benchmarking Linear Search
start_linear = timer()
res_linear = locate_card_linear(large_test_cards, target_query)
end_linear = timer()
print(f"Linear Search Runtime: {math.ceil((end_linear - start_linear)*1000)} ms | Found at: {res_linear}")

# 2. Benchmarking Binary Search
start_binary = timer()
res_binary = locate_card_binary(large_test_cards, target_query)
end_binary = timer()
print(f"Binary Search Runtime: {((end_binary - start_binary)*1000):.4f} ms | Found at: {res_binary}")

Linear Search Runtime: 583 ms | Found at: 9999998
Binary Search Runtime: 0.0756 ms | Found at: 9999998


## 7. Reusable Architecture: Generic Binary Search Engine
We can completely isolate the binary bounds checking logic by passing functional constraints as clean closures. This isolates the loop mechanics into a permanent, production-ready black box.

In [8]:
def binary_search(lo, hi, condition):
    """Generic structural container for executing binary range evaluations."""
    while lo <= hi:
        mid = (lo + hi) // 2
        result = condition(mid)
        if result == 'found':
            return mid
        elif result == 'left':
            hi = mid - 1
        else:
            lo = mid + 1
    return -1

def locate_card_generic(cards, query):
    """Succinct wrapper showcasing functional closure utilization."""
    def condition(mid):
        if cards[mid] == query:
            if mid > 0 and cards[mid - 1] == query:
                return 'left'
            return 'found'
        elif cards[mid] < query:
            return 'left'
        else:
            return 'right'
            
    return binary_search(0, len(cards) - 1, condition)

# Verify structural equivalence of our abstracted model
evaluate_test_cases(locate_card_generic, tests)

TEST CASE #0: PASSED | Runtime: 0.004 ms
TEST CASE #1: PASSED | Runtime: 0.005 ms
TEST CASE #2: PASSED | Runtime: 0.003 ms
TEST CASE #3: PASSED | Runtime: 0.003 ms
TEST CASE #4: PASSED | Runtime: 0.002 ms
TEST CASE #5: PASSED | Runtime: 0.002 ms
TEST CASE #6: PASSED | Runtime: 0.001 ms
TEST CASE #7: PASSED | Runtime: 0.002 ms
TEST CASE #8: PASSED | Runtime: 0.002 ms

SUMMARY: TOTAL: 9 | PASSED: 9 | FAILED: 0


## 8. Generalization Exercise: Find First and Last Position (LeetCode 34)
By subtly shifting the evaluation condition inside our generic search framework, we can seamlessly parse ascending items to return both boundary markers of a repeating segment.

In [9]:
def first_position(nums, target):
    def condition(mid):
        if nums[mid] == target:
            if mid > 0 and nums[mid - 1] == target:
                return 'left'
            return 'found'
        elif nums[mid] < target:
            return 'right' # Moving right updates bounds towards larger ascending keys
        else:
            return 'left'
    return binary_search(0, len(nums) - 1, condition)

def last_position(nums, target):
    def condition(mid):
        if nums[mid] == target:
            if mid < len(nums) - 1 and nums[mid + 1] == target:
                return 'right'
            return 'found'
        elif nums[mid] < target:
            return 'right'
        else:
            return 'left'
    return binary_search(0, len(nums) - 1, condition)

def first_and_last_position(nums, target):
    """Returns total coordinate boundaries as a tuple pair."""
    return first_position(nums, target), last_position(nums, target)

# Verification Sample (Ascending Array)
ascending_sample = [1, 2, 5, 5, 5, 5, 7, 9]
print(f"Boundaries for element 5: {first_and_last_position(ascending_sample, 5)}")

Boundaries for element 5: (2, 5)
